# OCTA 血管分割结果与标签对比

本 Notebook 加载训练阶段保存的 `best.pt`，在测试集上计算总体指标，并逐病例展示：

1. OCTA 输入图像
2. Ground Truth
3. 预测概率图
4. 二值预测结果
5. TP / FP / FN 误差分布
6. 标签和预测轮廓叠加

> 请先完成正式训练，或将 `RUN_DIR` 修改为已有实验结果目录。图内文字统一使用英文。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Patch
from torch.utils.data import DataLoader

from octa.data import OCTA500Dataset
from octa.engine import run_epoch
from octa.losses import BCEDiceLoss
from octa.model import UNet

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "octa_baseline":
    REPO_ROOT = PROJECT_ROOT.parent
else:
    REPO_ROOT = PROJECT_ROOT

DATA_ROOT = REPO_ROOT / "data" / "OCTA-500数据集"
RUN_DIR = REPO_ROOT / "octa_baseline" / "runs" / "unet_3mm_ilm_opl"
CHECKPOINT_PATH = RUN_DIR / "best.pt"

print(f"数据目录：{DATA_ROOT}")
print(f"实验目录：{RUN_DIR}")
print(f"模型文件：{CHECKPOINT_PATH}")

## 1. 加载最佳模型与测试集

模型结构和数据参数优先从 checkpoint 中读取，因此可用于 3 mm、6 mm 或不同投影的实验。

In [ ]:
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f"未找到 {CHECKPOINT_PATH}。请先运行训练，或在上一个单元格修改 RUN_DIR。"
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
saved_args = checkpoint.get("args", {})

scan_size = saved_args.get("scan_size", "3mm")
projection = saved_args.get("projection", "OCTA(ILM_OPL)")
target = saved_args.get("target", "GT_Capillary")
threshold = float(saved_args.get("threshold", 0.5))
base_channels = int(saved_args.get("base_channels", 32))

model = UNet(base=base_channels).to(device)
model.load_state_dict(checkpoint["model"])
model.eval()

test_dataset = OCTA500Dataset(
    root=DATA_ROOT,
    scan_size=scan_size,
    split="test",
    projection=projection,
    target=target,
    augment=False,
)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=0)

print(f"设备：{device}")
print(f"配置：{scan_size}, {projection} -> {target}, threshold={threshold}")
print(f"最佳 epoch：{checkpoint.get('epoch', 'unknown')}")
print(f"测试病例数：{len(test_dataset)}")

## 2. 完整测试集指标

这里使用固定阈值和全测试集累计混淆矩阵，计算 Dice、IoU、Precision、Recall 与 Specificity。

In [ ]:
test_metrics = run_epoch(
    model=model,
    loader=test_loader,
    criterion=BCEDiceLoss(),
    device=device,
    threshold=threshold,
    use_amp=device.type == "cuda",
    description="Test evaluation",
)

for name, value in test_metrics.items():
    print(f"{name:>12s}: {value:.6f}")

## 3. 逐病例可视化对比

可以直接修改 `CASE_IDS`。留空时默认展示测试集前 4 个病例。

误差图颜色：绿色为 TP，红色为 FP，蓝色为 FN，黑色为 TN。轮廓叠加图中绿色为标签轮廓，红色为预测轮廓。

In [ ]:
# 示例：CASE_IDS = ["10451", "10475", "10500"]
CASE_IDS = []
NUM_CASES = 4

if not CASE_IDS:
    CASE_IDS = test_dataset.ids[:NUM_CASES]

unknown_ids = sorted(set(CASE_IDS) - set(test_dataset.ids))
if unknown_ids:
    raise ValueError(f"以下病例不属于当前测试集：{unknown_ids}")
print("将展示病例：", CASE_IDS)

In [ ]:
def sample_metrics(prediction: np.ndarray, target_mask: np.ndarray) -> dict[str, float]:
    prediction = prediction.astype(bool)
    target_mask = target_mask.astype(bool)
    tp = np.logical_and(prediction, target_mask).sum()
    fp = np.logical_and(prediction, ~target_mask).sum()
    fn = np.logical_and(~prediction, target_mask).sum()
    eps = 1e-8
    return {
        "dice": (2 * tp) / (2 * tp + fp + fn + eps),
        "iou": tp / (tp + fp + fn + eps),
        "precision": tp / (tp + fp + eps),
        "recall": tp / (tp + fn + eps),
    }


def error_rgb(prediction: np.ndarray, target_mask: np.ndarray) -> np.ndarray:
    prediction = prediction.astype(bool)
    target_mask = target_mask.astype(bool)
    rgb = np.zeros((*prediction.shape, 3), dtype=np.float32)
    rgb[np.logical_and(prediction, target_mask)] = (0.0, 0.8, 0.0)  # TP
    rgb[np.logical_and(prediction, ~target_mask)] = (1.0, 0.0, 0.0)  # FP
    rgb[np.logical_and(~prediction, target_mask)] = (0.0, 0.3, 1.0)  # FN
    return rgb


index_by_id = {case_id: index for index, case_id in enumerate(test_dataset.ids)}
records = []
with torch.inference_mode():
    for case_id in CASE_IDS:
        sample = test_dataset[index_by_id[case_id]]
        image_tensor = sample["image"].unsqueeze(0).to(device)
        probability = torch.sigmoid(model(image_tensor))[0, 0].cpu().numpy()
        image = sample["image"][0].numpy()
        ground_truth = sample["mask"][0].numpy().astype(bool)
        prediction = probability >= threshold
        records.append((case_id, image, ground_truth, probability, prediction))

figure, axes = plt.subplots(len(records), 6, figsize=(19, 3.4 * len(records)), squeeze=False)
for row, (case_id, image, ground_truth, probability, prediction) in enumerate(records):
    values = sample_metrics(prediction, ground_truth)
    panels = (image, ground_truth, probability, prediction, error_rgb(prediction, ground_truth))
    titles = (
        "OCTA input",
        "Ground truth",
        "Probability map",
        f"Prediction (t={threshold:.2f})",
        "Error map",
    )
    for column, (panel, title) in enumerate(zip(panels, titles)):
        cmap = None if column == 4 else "gray"
        axes[row, column].imshow(panel, cmap=cmap, vmin=0, vmax=1)
        axes[row, column].set_title(f"Case {case_id} | {title}", fontsize=9)
        axes[row, column].axis("off")

    overlay_axis = axes[row, 5]
    overlay_axis.imshow(image, cmap="gray", vmin=0, vmax=1)
    overlay_axis.contour(ground_truth.astype(float), levels=[0.5], colors=["lime"], linewidths=0.8)
    overlay_axis.contour(prediction.astype(float), levels=[0.5], colors=["red"], linewidths=0.8)
    overlay_axis.set_title(
        f"Contour overlay\nDice={values['dice']:.3f}, IoU={values['iou']:.3f}", fontsize=9
    )
    overlay_axis.axis("off")

legend_items = [
    Patch(facecolor=(0.0, 0.8, 0.0), label="TP"),
    Patch(facecolor=(1.0, 0.0, 0.0), label="FP"),
    Patch(facecolor=(0.0, 0.3, 1.0), label="FN"),
    Patch(facecolor=(0.0, 0.0, 0.0), label="TN"),
]
figure.legend(handles=legend_items, loc="lower center", ncol=4, frameon=False)
figure.suptitle(f"OCTA Vessel Segmentation: Prediction vs Ground Truth ({scan_size})", fontsize=15)
figure.tight_layout(rect=(0, 0.04, 1, 0.97))
plt.show()

## 4. 训练曲线（可选）

如果实验目录中存在 `history.csv`，下面会展示训练/验证 Loss、Dice 和 IoU 曲线。

In [ ]:
import csv

history_path = RUN_DIR / "history.csv"
if history_path.is_file():
    with history_path.open(encoding="utf-8") as file:
        history = list(csv.DictReader(file))
    epochs = [int(row["epoch"]) for row in history]
    figure, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    for axis, metric in zip(axes, ("loss", "dice", "iou")):
        axis.plot(epochs, [float(row[f"train_{metric}"]) for row in history], label="Train")
        axis.plot(epochs, [float(row[f"val_{metric}"]) for row in history], label="Validation")
        axis.set_title(metric.upper())
        axis.set_xlabel("Epoch")
        axis.grid(alpha=0.25)
        axis.legend()
    plt.show()
else:
    print(f"未找到训练历史：{history_path}")